In [1]:
import os, subprocess

REPO_URL = "https://github.com/tongyuguo/HelpHerInvest.git"
REPO_DIR = "HelpHerInvest"
data_dir = os.path.join(REPO_DIR, "Data")

def clone_or_pull():
    if os.path.isdir(os.path.join(REPO_DIR, ".git")):
        subprocess.run(["git", "-C", REPO_DIR, "pull"])
    else:
        subprocess.run(["git", "clone", REPO_URL])

clone_or_pull()

Already up to date.


In [2]:
import pandas as pd
import zipfile

zip_path = "HelpHerInvest/Data/stock_symbols_new.csv.zip"

with zipfile.ZipFile(zip_path) as z:
    df = pd.read_csv(z.open("stock_symbols_new.csv"))

df.head()

/tmp/ipykernel_2392300/1261020715.py:7: DtypeWarning: Columns (54,204,212,213,214,220,221,222,223) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(z.open("stock_symbols_new.csv"))


,symbol,company_name,address1,city,state,zip,country,phone,website,industry,...,address3,morningStarOverallRating,morningStarRiskRating,annualReportExpenseRatio,lastCapGain,annualHoldingsTurnover,prevTicker,tickerChangeDate,newListingDate,delistingDate
0,NVDA,NVIDIA CORP,2788 San Tomas Expressway,Santa Clara,CA,95051,United States,408 486 2000,https://www.nvidia.com,Semiconductors,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,GOOGL,Alphabet Inc.,1600 Amphitheatre Parkway,Mountain View,CA,94043,United States,650-253-0000,https://abc.xyz,Internet Content & Information,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,AAPL,Apple Inc.,One Apple Park Way,Cupertino,CA,95014,United States,(408) 996-1010,https://www.apple.com,Consumer Electronics,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,MSFT,MICROSOFT CORP,One Microsoft Way,Redmond,WA,98052-6399,United States,425 882 8080,https://www.microsoft.com,Software - Infrastructure,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,AMZN,AMAZON COM INC,410 Terry Avenue North,Seattle,WA,98109-5210,United States,206 266 1000,https://www.amazon.com,Internet Retail,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:

import numpy as np
import re
import ast
import unicodedata
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS


# Drop rows with missing symbol (ID)
df = df.dropna(subset=["symbol"]).copy()

# Convert common placeholder strings to NA
PLACEHOLDERS = {"[]": np.nan, "None": np.nan, "nan": np.nan, "NaN": np.nan, "": np.nan}
for col in df.columns:
    if df[col].dtype == "object":
        df[col] = df[col].replace(PLACEHOLDERS)


# Parse companyOfficers into plain text
def parse_officers(officers_str):
    if pd.isna(officers_str):
        return []
    try:
        parsed = ast.literal_eval(officers_str)
        return parsed if isinstance(parsed, list) else []
    except Exception:
        return []

def officers_to_text(officers):
    parts = []
    for o in officers:
        if not isinstance(o, dict):
            continue
        name = o.get("name")
        title = o.get("title")
        if name and title:
            parts.append(f"{name} - {title}")
        elif name:
            parts.append(str(name))
        elif title:
            parts.append(str(title))
    return "; ".join(parts)

df["companyOfficers_parsed"] = df["companyOfficers"].apply(parse_officers)
df["companyOfficers_text"] = df["companyOfficers_parsed"].apply(officers_to_text)

# Build the raw NLP document per company
def join_text(*items):
    items = [x for x in items if isinstance(x, str) and x.strip()]
    return " ".join(items).strip()

df["document_raw"] = df.apply(
    lambda r: join_text(
        r.get("company_name"),
        r.get("sector"),
        r.get("industry"),
        r.get("longBusinessSummary"),
        r.get("companyOfficers_text"),
    ),
    axis=1
)

# Keep only rows with some text
df = df[df["document_raw"].str.len().fillna(0) > 0].copy()

# Text cleaning for NLP
URL_RE = re.compile(r"https?://\S+|www\.\S+", re.IGNORECASE)
EMAIL_RE = re.compile(r"\b[\w\.-]+@[\w\.-]+\.\w+\b")
HTML_RE = re.compile(r"<.*?>")
NON_WORD_RE = re.compile(r"[^a-z\s]") 
MULTISPACE_RE = re.compile(r"\s+")


def clean_text_minimal(text: str) -> str:
    if pd.isna(text):
        return ""

    # Unicode normalize
    text = unicodedata.normalize("NFKC", str(text))

    # Remove HTML, URLs, emails
    text = re.sub(HTML_RE, " ", text)
    text = re.sub(URL_RE, " ", text)
    text = re.sub(EMAIL_RE, " ", text)

    # Lowercase + whitespace normalize
    text = text.lower()
    text = re.sub(MULTISPACE_RE, " ", text).strip()
    return text

def clean_text_for_tfidf(text: str, remove_stopwords: bool = True) -> str:
    text = clean_text_minimal(text)
    text = re.sub(NON_WORD_RE, " ", text)
    text = re.sub(MULTISPACE_RE, " ", text).strip()

    if remove_stopwords:
        tokens = [t for t in text.split() if t not in ENGLISH_STOP_WORDS and len(t) > 2]
        text = " ".join(tokens)

    return text

# Choose one:
#df["document_clean_minimal"] = df["document_raw"].apply(clean_text_minimal)
df["document_clean_tfidf"] = df["document_raw"].apply(clean_text_for_tfidf)

# Optional: drop documents that became too short after cleaning
#df = df[df["document_clean_minimal"].str.len() >= 30].copy()



# 5) Save cleaned dataset for modeling

cols_to_save = [
    "symbol",
    "company_name",
    "sector",
    "industry",
    "document_raw",
#    "document_clean_minimal",
    "document_clean_tfidf",
]

df[cols_to_save].to_csv(os.path.join(data_dir, "nlp_clean_stock_symbols.csv"), index=False)
print("Saved:", "nlp_clean_stock_symbols.csv", "rows:", len(df))

/tmp/ipykernel_2392300/4247770155.py:15: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].replace(PLACEHOLDERS)


Saved: nlp_clean_stock_symbols.csv rows: 10283


In [4]:
def can_push():
    result = subprocess.run(
        ["git", "-C", REPO_DIR, "push", "--dry-run"],
        capture_output=True, text=True
    )
    return result.returncode == 0

def commit_and_push(msg="update"):
    subprocess.run(["git", "-C", REPO_DIR, "add", "-A"])
    subprocess.run(["git", "-C", REPO_DIR, "commit", "-m", msg])
    subprocess.run(["git", "-C", REPO_DIR, "push"])

if can_push():
    commit_and_push("update notebook / data")
else:
    print("Not a collaborator — skipping push")

[main 6c1a3bc] update notebook / data
 1 file changed, 10292 insertions(+)
 create mode 100644 Data/nlp_clean_stock_symbols.csv


To https://github.com/tongyuguo/HelpHerInvest.git
   6d6aa35..6c1a3bc  main -> main
